In [0]:
from pyspark.sql.functions import *
import time
# Environment selection as text
dbutils.widgets.text(
    name="environment",
    defaultValue="fq_dev",
    label="Environment"
)

# Source selection as text
dbutils.widgets.text(
    name="source",
    defaultValue="posist",
    label="Source"
)

# Domain selection as text
dbutils.widgets.text(
    name="domain",
    defaultValue="sales_bills_2",
    label="Domain"
)


environment = dbutils.widgets.get("environment")
source = dbutils.widgets.get("source")
domain = dbutils.widgets.get("domain")

# Get external location URLs
bronze_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_bronze`"
).select("url").collect()[0][0]

silver_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_silver`"
).select("url").collect()[0][0]

gold_path = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_gold`"
).select("url").collect()[0][0]

checkpoint = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_checkpoint`"
).select("url").collect()[0][0]

staging = spark.sql(
    f"DESCRIBE EXTERNAL LOCATION `{environment}_extloc_staging`"
).select("url").collect()[0][0]
time.sleep(20)

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS fq_dev_catalog.bronze.sales_bills_2 (
#   businessDate DATE
# )
# USING DELTA
# TBLPROPERTIES (
#   delta.enableChangeDataFeed = true,
#   delta.columnMapping.mode = 'name'
# );

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    col,
    regexp_replace,
    to_date,
)
import time
 
def start_stream_restart_on_schema_evolution():
    while True:
        try:
            q = (
                spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "json")
                    .option("cloudFiles.allowOverwrites", "true")
                    # .option("multiLine", "true")
                    .option(
                        "cloudFiles.schemaLocation",
                        f"{checkpoint}/{source}/{domain}/infschema_bills_2"
                    )
                    .option("cloudFiles.inferColumnTypes", "true")
                    .option("cloudFiles.schemaEvolutionMode", "rescue")
                    .load(f"{staging}/FoodQuest/{{ALBAIK,Dennys}}/*/bills.json")
 
                    # -------- METADATA --------
                    .withColumn("ingestion_ts", current_timestamp())
                    .withColumn(
                        "file_path",
                        regexp_replace(col("_metadata.file_path"), "%20", " ")
                    )
 
                    # -------- BUSINESS DATE --------
                    .withColumn(
                        "businessDate",
                        to_date(col("businessDate"))   # cast to DATE
                    )
 
                    # -------- WRITE STREAM --------
                    .writeStream
                    .format("delta")  
                    .trigger(availableNow=True)
                    .outputMode("append")
                    .queryName(f"{domain}WriteStream")
                    .option("delta.enableChangeDataFeed", "true")
                    .partitionBy("businessDate")
                    .option('mergeSchema', 'true')
                    .toTable(
                        f"`{environment}_catalog`.`bronze`.`{domain}`",
                            checkpointLocation=f'{checkpoint}/{source}/{domain}/chkpoint_bills_2',
                            mergeSchema=True
                    )
            )
 
            q.awaitTermination()
            print("Streaming write completed successfully.")
            return q
 
        except BaseException as e:
            if "UNKNOWN_FIELD_EXCEPTION" in str(e):
                print(f"Going to restart stream after schema change:\n{e}")
            else:
                raise e
 
# 🔹 CALL
start_stream_restart_on_schema_evolution()
time.sleep(20)

In [0]:
# %sql
# SELECT * FROM fq_dev_catalog.bronze.sales_bills_2 WHERE businessDate = '2026-01-15';

In [0]:
for query in spark.streams.active:
    print(f"Name: {query.name}, Id: {query.id}, Status: {query.status['message']}")

In [0]:
for query in spark.streams.active:
    query.stop()